# Initialization

In [0]:

import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import trim, col

# Read from the Bronze table

In [0]:

df = spark.table("workspace.bronze.crm_cust_info")

# Silver Transformations


## Trimming

In [0]:

for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))
     

## Normalization

In [0]:

df = (
    df
    .withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    )
    .withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

##   Remove Records with Missing Customer ID

In [0]:

df = df.filter(col("cst_id").isNotNull())

## Renaming Columns

In [0]:
RENAME_MAP = {
    "cst_id": "customer_id",
    "cst_key": "customer_key",
    "cst_firstname": "customer_firstname",
    "cst_lastname": "customer_lastname",
    "cst_marital_status": "customer_marital_status",
    "cst_gndr": "customer_gender",
    "cst_create_date": "customer_create_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

### Sanity checks of dataframe


In [0]:
df.limit(10).display()

customer_id,customer_key,customer_firstname,customer_lastname,customer_marital_status,customer_gender,customer_create_date
11000,AW00011000,Jon,Yang,Married,Male,2025-10-06
11001,AW00011001,Eugene,Huang,Single,Male,2025-10-06
11002,AW00011002,Ruben,Torres,Married,Male,2025-10-06
11003,AW00011003,Christy,Zhu,Single,Female,2025-10-06
11004,AW00011004,Elizabeth,Johnson,Single,Female,2025-10-06
11005,AW00011005,Julio,Ruiz,Single,Male,2025-10-06
11006,AW00011006,Janet,Alvarez,Single,Female,2025-10-06
11007,AW00011007,Marco,Mehta,Married,Male,2025-10-06
11008,AW00011008,Rob,Verhoff,Single,Female,2025-10-06
11009,AW00011009,Shannon,Carlson,Single,Male,2025-10-06


# Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.silver.crm_customers")
     

# Sanity checks of silver table

In [0]:
%sql
SELECT * FROM workspace.silver.crm_customers LIMIT 10

customer_id,customer_key,customer_firstname,customer_lastname,customer_marital_status,customer_gender,customer_create_date
11000,AW00011000,Jon,Yang,Married,Male,2025-10-06
11001,AW00011001,Eugene,Huang,Single,Male,2025-10-06
11002,AW00011002,Ruben,Torres,Married,Male,2025-10-06
11003,AW00011003,Christy,Zhu,Single,Female,2025-10-06
11004,AW00011004,Elizabeth,Johnson,Single,Female,2025-10-06
11005,AW00011005,Julio,Ruiz,Single,Male,2025-10-06
11006,AW00011006,Janet,Alvarez,Single,Female,2025-10-06
11007,AW00011007,Marco,Mehta,Married,Male,2025-10-06
11008,AW00011008,Rob,Verhoff,Single,Female,2025-10-06
11009,AW00011009,Shannon,Carlson,Single,Male,2025-10-06
